# Adaptive QA: 10-question setup check

This notebook checks the model, data preparation, retrieval and runtime on a small QANTA 2021 sample before the 50-question pilot. It uses Qwen2.5-0.5B-Instruct with BM25 retrieval. Repeated answers provide a consistency signal; agreement alone does not show that an answer is correct. Confidence calibration and controller training are outside this setup check.

## Project environment

Google Drive stores the project files, cached responses and results between Colab sessions. The following cells connect Drive, load the project source if it is missing, and install the required packages. Existing project files are reused.


In [ ]:
from google.colab import drive, files
from pathlib import Path, PurePosixPath
import os, sys, json, zipfile, shutil, subprocess, time, datetime, platform
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/AdaptiveQA2021_consistency/adaptive-confidence-agent')
print('Persistent working folder:', PROJECT)


In [ ]:
# First session only: upload stage2-colab-source.zip created from your committed local repository.
# Existing source is reused. It is never overwritten by a later upload.
if not (PROJECT/'src/adaptive_qa').exists():
    uploaded=files.upload()
    if len(uploaded)!=1: raise ValueError('Upload exactly one project source ZIP')
    archive=next(iter(uploaded))
    planned=[]
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            if entry.is_dir(): continue
            parts=PurePosixPath(entry.filename).parts
            if not parts or parts[0] not in ('adaptive-qa','adaptive-confidence-agent') or '..' in parts:
                raise ValueError('Expected a source ZIP with adaptive-qa/ or adaptive-confidence-agent/ at its root')
            rel=PurePosixPath(*parts[1:])
            if not rel.parts: continue
            allowed=rel.parts[0] in ('src','scripts','configs','notebooks') or len(rel.parts)==1
            allowed=allowed or rel.parts[:2]==('data','fixtures')
            if not allowed or '.git' in rel.parts or '__pycache__' in rel.parts or rel.name.startswith('.env'):
                continue
            target=PROJECT/str(rel)
            if target.exists(): raise FileExistsError(f'Existing file: {target}. Preserve it and choose a new working folder.')
            if not target.resolve().is_relative_to(PROJECT.resolve()): raise ValueError('Unsafe archive path')
            planned.append((target,z.read(entry)))
    if not any(str(p).endswith('src/adaptive_qa/pilot.py') for p,_ in planned):
        raise ValueError('This ZIP lacks the current consistency pilot update')
    for target,data in planned:
        target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(data)
else:
    print('Reusing existing Drive source; no ZIP was applied.')
if not (PROJECT/'src/adaptive_qa/pilot.py').exists():
    raise ValueError('Old project found. Preserve it and use a new working folder for this update.')
os.chdir(PROJECT)
print('Working directory:', Path.cwd())


In [ ]:
os.chdir(PROJECT)
subprocess.run([sys.executable,'-m','pip','install','-e','.[gpu,data]'],check=True)


## Run logs and configuration

The logging functions save each script's command, output, elapsed time and completion status. They also retain usage records produced during the run. The selected configuration defines the 10-question experiment and its model, retrieval and sampling limits.


In [ ]:
LOGS=PROJECT/'results/logs';LOGS.mkdir(parents=True,exist_ok=True)
def run_script(script,*args):
    """Log each invocation, including failure status and usage snapshots."""
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    folder=LOGS/stamp;folder.mkdir()
    command=[sys.executable,'-u',script,*map(str,args)]
    start=time.time();returncode=None
    try:
        with (folder/'stdout.log').open('w',encoding='utf-8') as log:
            proc=subprocess.Popen(command,cwd=PROJECT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
            for line in proc.stdout:
                print(line,end='');log.write(line);log.flush()
            returncode=proc.wait()
    finally:
        record={'command':command,'started_utc':stamp,'wall_seconds':time.time()-start,'returncode':returncode,
                'note':'A disconnected runtime may prevent final logging. Record interruptions manually too.'}
        (folder/'invocation.json').write_text(json.dumps(record,indent=2)+'\n')
        for path in (PROJECT/'results/runs').glob('*/*usage.json'):
            if path.stat().st_mtime>=start:
                shutil.copy2(path,folder/(path.parent.name+'__'+path.name))
    if returncode!=0: raise RuntimeError(f'Command failed or was interrupted; see {folder}')
    return folder

def pipeline(command,config,execute=False):
    args=[command,'--config',config]
    if execute:
        import torch
        if not torch.cuda.is_available(): raise RuntimeError('A free CUDA GPU is needed for this config; do not buy an upgrade.')
        args.append('--execute-model')
    return run_script('scripts/run.py',*args)


In [ ]:
CONFIG='configs/colab_qanta2021_pilot10.yaml'


## QANTA 2021 question selection

The data preparation script downloads the QANTA release dated 2021.12.20 and creates a fixed selection of 50 questions. The 10-question setup sample is drawn from that same selection. Saving the selection allows later sessions to use the same questions and split assignments.


In [ ]:
run_script('scripts/prepare_qanta_public.py','download')
if not (PROJECT/'data/processed/qanta2021/selection.json').exists():
    run_script('scripts/prepare_qanta_public.py','select')
else:
    print('Existing selection preserved.')


## Answer review

The review CSV pairs each selected question with its original answerline and proposed accepted answers. The `accepted_answers_json` field records the answer variants used for scoring, and `approved` records the review status. The question text, source answerline and split assignments remain unchanged. This cell exports the CSV for review before model inference.


In [ ]:
review=PROJECT/'data/processed/qanta2021/answer_review.csv'
files.download(str(review))


## Frozen questions and splits

The uploaded CSV supplies the reviewed answers for the experiment. The freeze script checks that all 50 questions are present, the source fields are unchanged, and every row has an approval and a valid answer list. It then saves the 10-question and 50-question datasets with their question-level splits. The guard prevents this cell from replacing an already frozen selection.


In [ ]:
if (PROJECT/'data/processed/qanta2021/frozen_manifest.json').exists():
    raise RuntimeError('Already frozen. Preserve the reviewed selection and skip this upload cell.')
uploaded=files.upload()
if len(uploaded)!=1: raise ValueError('Upload exactly the reviewed answer_review.csv')
name,data=next(iter(uploaded.items()))
if not name.lower().endswith('.csv'): raise ValueError('Expected CSV')
review.write_bytes(data)
run_script('scripts/prepare_qanta_public.py','freeze')


## Wikipedia evidence corpus

The evidence corpus contains passages from up to 500 Wikipedia articles selected using QANTA training-set titles. Article text is cached and divided into passages of up to 150 words, with source URLs and revision information retained. BM25 searches these passages during retrieval. The saved Wikipedia revisions define the evidence snapshot; they are separate from the QANTA 2021 release. The contact value identifies the project in Wikipedia API requests.


In [ ]:
from getpass import getpass
contact=getpass('Contact email or project URL for Wikipedia: ')
# Avoid recording contact details in the command logger.
subprocess.run([sys.executable,'scripts/build_wikipedia_corpus.py','--contact',contact],check=True)
del contact


## Model version and experiment limits

The model is `Qwen/Qwen2.5-0.5B-Instruct`. Its revision is pinned so repeated runs use the same model version. Each question is evaluated at half and full length, with two generations per observation and at most one retrieval step. The preparation and estimation calls check the inputs and planned workload before inference. The configuration sets a budget of 100 new model calls for the setup run.


In [ ]:
run_script('scripts/pin_colab_model.py')
import yaml
config=yaml.safe_load((PROJECT/CONFIG).read_text())
assert config['calibration']['enabled'] is False
assert config['controller']['actions']==['RETRIEVE'] and config['controller']['max_steps']==1
assert config['model']['samples']==2 and config['evaluation']['include_wait'] is False
print('Pinned model:',config['model']['name'],config['model']['revision'])


In [ ]:
pipeline('prepare',CONFIG)
pipeline('estimate',CONFIG)


## Hardware record and baseline

The hardware record captures the assigned GPU, total GPU memory, Python version and package versions. The baseline generates answers without retrieved evidence and provides a reference for the retrieval comparison. Cached outputs and usage records help distinguish reused responses from new computation.


In [ ]:
import torch, importlib.metadata
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU assigned. Continue CPU work or try a free GPU later.')
info={'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
      'python':platform.python_version(),'gpu':torch.cuda.get_device_name(0),
      'vram_gib':torch.cuda.get_device_properties(0).total_memory/2**30,
      'packages':{p:importlib.metadata.version(p) for p in ('torch','transformers','numpy','scikit-learn','PyYAML')}}
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
(LOGS/f'hardware_{stamp}.json').write_text(json.dumps(info,indent=2)+'\n')
print(json.dumps(info,indent=2))


In [ ]:
pipeline('baseline',CONFIG,execute=True)


## Retrieval comparison and pilot report

The collection step generates answers before and after one retrieval step, using up to three evidence passages. The verification step checks the collected outputs, and the report summarizes answer accuracy, valid output format, consistency and computational usage. The results are separated by question length. Repeated wrong answers can be consistent, so consistency and correctness are reported separately. These are pilot observations for checking the workflow.


In [ ]:
pipeline('collect',CONFIG,execute=True)


In [ ]:
run_script('scripts/verify_collection.py','--config',CONFIG)
pipeline('pilot-report',CONFIG)


In [ ]:
out=PROJECT/'results/runs'/config['experiment_id']
print((out/'pilot_report.md').read_text())
print(json.dumps(json.loads((out/'pilot_report.json').read_text()),indent=2))


## Saved configuration and run archive

The final cells export the pinned experiment configuration and create a private archive of the project files, including processed data, evidence, cached responses, logs and results. Downloaded model weights and the original QANTA release files are excluded from the archive. This backup preserves the experiment records separately from the code repository.


In [ ]:
# Save the pinned config to your PC; compare it before updating the local repository.
files.download(str(PROJECT/CONFIG))


In [ ]:
# Keep a second private copy on your PC; do not commit this entire backup to GitHub.
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
backup=Path('/content')/f'adaptiveqa-private-backup-{stamp}.zip'
with zipfile.ZipFile(backup,'w',zipfile.ZIP_DEFLATED) as z:
    for path in PROJECT.rglob('*'):
        if not path.is_file(): continue
        rel=path.relative_to(PROJECT)
        if any(part in ('.git','.venv','__pycache__','model_cache') for part in rel.parts): continue
        if path.name.startswith('.env') or path.suffix in ('.safetensors','.bin','.pt','.zip'): continue
        if path.name in ('qanta.train.2021.12.20.json','qanta.dev.2021.12.20.json'): continue
        z.write(path,Path('adaptive-confidence-agent')/rel)
print('Backup:',backup,'bytes:',backup.stat().st_size)
files.download(str(backup))
